In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import dataset
import network
import train_network

In [ ]:
device = 'cpu'
network_shape = [3, 100, 100, 100, 2]
scheduler = train_network.Scheduler()
device

In [ ]:
net = network.Network(network_shape, scheduler).to(device)
checkpoint = torch.load('network_final.pt', map_location=device)
net.load_state_dict(checkpoint['weights'])

In [ ]:
def ddim_timestep_diffuse(net, npts,  taus):
    locations = np.zeros((npts, 2, len(taus)+1))
    normal_distribution = np.random.normal(0, 1, (npts, 2))
    locations[:, :, -1] = normal_distribution
    
    with torch.no_grad():
        xs = torch.tensor(normal_distribution, dtype=torch.float32).to(device)
        tau_minus_1 = scheduler.T-1
        for i, tau in enumerate(taus):
            e_theta = net.forward(xs, int(tau), return_bitstrings=False)
            atm1 = scheduler.alphabars[tau]
            at = scheduler.alphabars[tau_minus_1]
            x_new = np.sqrt(atm1 / at)*(xs - np.sqrt(1-at)*e_theta) + np.sqrt(1-atm1)*e_theta
            locations[:, :, len(taus)-i-1] = x_new.detach().cpu().numpy()
            xs = x_new
            tau_minus_1 = tau
            
    return locations
    

In [ ]:
# times = [i for i in range(900,0,-100)] + [i for i in range(90,0,-10)] +[i for i in range(9,0,-1)]
# times = [i for i in range(900,0,-100)] + [i for i in range(90,0,-10)] +[0]
# times = np.linspace(999,0,50).astype(int)
# times = [i for i in range(900,0,-100)] + [2]
# times = [900,800,700,600,500,400,300,200,100,90,80,70,60,50,40,30,20,10]
# times = [800,600,400,200,250,100,80,60,50,40,30,20,10,1]
times = [800, 500, 300, 200, 100, 90, 70, 50, 30, 20, 10, 9, 8, 7, 6, 5, 4, 3, 2, 1]
n=1000
pts = ddim_timestep_diffuse(net, n, times)
colors = np.random.uniform(0,1,[n,3])
print(times)

In [ ]:
i = 0
plt.scatter(pts[:,0,i],pts[:,1,i],c=colors, s=2)
plt.ylim([-2, 2])
plt.xlim([-2, 2])
plt.gca().set_aspect('equal')
plt.title("Using Subprocess")
num_trajectories = n
for k in range(num_trajectories):
    plt.plot(pts[k,0,i:],pts[k,1,i:],c=colors[k,:],alpha=0.1)
plt.savefig("ddim_reduced_steps.jpg")